In [1]:
# --- iPython Config --- #
from IPython import get_ipython
if 'IPython.extensions.autoreload' not in get_ipython().extension_manager.loaded:
    get_ipython().run_line_magic('load_ext', 'autoreload')
else:
    get_ipython().run_line_magic('reload_ext', 'autoreload')
%autoreload 2

# --- System and Path --- #
import os
import sys
REPO_PATH = os.path.abspath(os.path.join('..'))
if REPO_PATH not in sys.path:
    sys.path.append(REPO_PATH)
import warnings
warnings.filterwarnings("ignore")

# --- Data Manipulation --- #
import pandas as pd
import numpy as np

In [2]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-

"""
2110572 Take-Home Exam: CU Automatic Short Answer Scoring (Advanced Example)
Author: Your Name

Description:
  End-to-end script that:
    1) Reads train, test, and sample_submission data.
    2) Combines (question + answer) text.
    3) Uses a pretrained Thai BERT (WangChanBERTa) from Hugging Face
       to embed the text into dense vectors.
    4) Trains an XGBoost regressor on these embeddings to predict scores.
    5) Evaluates performance via cross-validation and saves submission.csv

Usage:
  - Ensure you have installed:
      pip install transformers xgboost
  - Place train.csv, test.csv, sample_submission.csv in ./data/ by default.
  - Run this script (python advanced_shorts_scoring.py).
  - Check console logs for CV MSE and final submission generation.
"""

import os
import numpy as np
import pandas as pd
from tqdm import tqdm  # progress bar (pip install tqdm)

import torch
from transformers import AutoTokenizer, AutoModel

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import xgboost as xgb


###############################################################################
# 1) Data Reading
###############################################################################
def read_data():
    """Reads train.csv, test.csv, sample_submission.csv from ./data/."""
    data_dir = os.path.join(REPO_PATH, "data")
    train_path = os.path.join(data_dir, "train.csv")
    test_path = os.path.join(data_dir, "test.csv")
    sample_sub_path = os.path.join(data_dir, "sample_submission.csv")

    for path_ in [train_path, test_path, sample_sub_path]:
        if not os.path.exists(path_):
            raise FileNotFoundError(f"Missing file: {path_}")

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)
    sub_df = pd.read_csv(sample_sub_path)

    return train_df, test_df, sub_df


###############################################################################
# 2) Preprocessing and Thai BERT Embeddings
###############################################################################
def combine_text(question, answer):
    """Simple text combination: question + answer."""
    q = question if isinstance(question, str) else ""
    a = answer if isinstance(answer, str) else ""
    return q + " " + a

class ThaiBERTEmbedder:
    """
    Converts text into embeddings using a Thai BERT model from Hugging Face.
    For example: "airesearch/wangchanberta-base-att-spm-uncased"
    """
    def __init__(self, model_name="airesearch/wangchanberta-base-att-spm-uncased", device=None):
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        print(f"Loading tokenizer/model for {model_name} on device = {self.device}")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModel.from_pretrained(model_name).to(self.device)
        self.model.eval()  # inference mode

    def encode(self, text_list):
        """
        Args:
            text_list (List[str]): List of raw texts to encode.
        Returns:
            np.ndarray of shape (len(text_list), hidden_dim),
            containing BERT embeddings.
        """
        embeddings = []
        # We'll batch the texts for speed, or do single for simplicity.
        batch_size = 16
        for i in range(0, len(text_list), batch_size):
            batch_texts = text_list[i : i + batch_size]
            inputs = self.tokenizer(
                batch_texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=128
            ).to(self.device)
            with torch.no_grad():
                outputs = self.model(**inputs)
            # Let's take the [CLS] token embedding or the first token
            cls_emb = outputs.last_hidden_state[:, 0, :]  # shape = [batch_size, hidden_dim]
            embeddings.append(cls_emb.cpu().numpy())
        return np.concatenate(embeddings, axis=0)


def preprocess_data(train_df, test_df, embedder):
    """Combine Q+A, produce embeddings for train and test."""
    train_df["text"] = train_df.apply(lambda r: combine_text(r["question"], r["answer"]), axis=1)
    test_df["text"]  = test_df.apply(lambda r: combine_text(r["question"], r["answer"]), axis=1)

    X_train_texts = train_df["text"].tolist()
    X_test_texts  = test_df["text"].tolist()
    y_train       = train_df["score"].values

    print("Encoding training text to embeddings...")
    X_train_emb   = embedder.encode(X_train_texts)
    print("Encoding test text to embeddings...")
    X_test_emb    = embedder.encode(X_test_texts)

    return X_train_emb, y_train, X_test_emb


###############################################################################
# 3) Cross-Validation with XGBoost
###############################################################################
def run_cross_val(X_train_emb, y_train, n_splits=5):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)
    mses = []

    for train_idx, val_idx in kf.split(X_train_emb):
        X_tr, X_val = X_train_emb[train_idx], X_train_emb[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        # XGBoost regressor (feel free to tune hyperparams)
        model = xgb.XGBRegressor(
            n_estimators=100,
            max_depth=8,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=42,
            n_jobs=-1,
            tree_method="auto",
        )
        model.fit(X_tr, y_tr)

        y_pred_val = model.predict(X_val)
        fold_mse = mean_squared_error(y_val, y_pred_val)
        mses.append(fold_mse)

    return mses


###############################################################################
# 4) Final Training & Prediction
###############################################################################
def train_final_model(X_train_emb, y_train):
    """Train final model on full training data and return the fitted XGBRegressor."""
    model = xgb.XGBRegressor(
        n_estimators=100,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1,
        tree_method="auto",
    )
    model.fit(X_train_emb, y_train)
    return model


def predict_and_save(model, X_test_emb, sub_df, output_name="submission.csv"):
    """Generate predictions on test embeddings, save to submission CSV."""
    predictions = model.predict(X_test_emb)
    sub_df["score"] = predictions
    sub_df.to_csv(output_name, index=False)
    print(f"Submission file saved: {output_name}")


###############################################################################
# Main Entry Point
###############################################################################
def main():
    print("=== 1) Reading data ===")
    train_df, test_df, sub_df = read_data()

    print("=== 2) Initializing Thai BERT embedder ===")
    embedder = ThaiBERTEmbedder(
        model_name="airesearch/wangchanberta-base-att-spm-uncased",
        device=None  # auto-detect
    )

    print("=== 3) Preprocessing & Embedding Data ===")
    X_train_emb, y_train, X_test_emb = preprocess_data(train_df, test_df, embedder)

    print("=== 4) Cross-validation ===")
    mses = run_cross_val(X_train_emb, y_train, n_splits=5)
    print("Fold MSEs:", mses)
    print("Mean CV MSE:", np.mean(mses))

    print("=== 5) Final training on full data ===")
    final_model = train_final_model(X_train_emb, y_train)

    print("=== 6) Predicting & Saving Submission ===")
    predict_and_save(final_model, X_test_emb, sub_df, output_name="submission.csv")

    print("All done! Check submission.csv for results.")


if __name__ == "__main__":
    main()


=== 1) Reading data ===
=== 2) Initializing Thai BERT embedder ===
Loading tokenizer/model for airesearch/wangchanberta-base-att-spm-uncased on device = cpu
=== 3) Preprocessing & Embedding Data ===
Encoding training text to embeddings...
Encoding test text to embeddings...
=== 4) Cross-validation ===
Fold MSEs: [np.float64(2.5177351899848985), np.float64(2.3364499730502986), np.float64(2.8600058692563115), np.float64(2.213443652810631), np.float64(3.330372971357595)]
Mean CV MSE: 2.651601531291947
=== 5) Final training on full data ===
=== 6) Predicting & Saving Submission ===
Submission file saved: submission.csv
All done! Check submission.csv for results.
